# Chunking walkthroughCanonicalDocument -> StructureAwareRecursiveChunker -> DocumentChunk[] -> PostgreSQL

## 0. Setup

In [1]:
import sys, os, json, logging
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from knowledgeos.logging_setup import configure_logging
configure_logging()
r = logging.getLogger(); r.handlers.clear()
h = logging.StreamHandler(sys.stdout)
h.setFormatter(logging.Formatter("%(levelname)-7s %(name)-20s %(message)s"))
r.addHandler(h); r.setLevel(logging.INFO)

from knowledgeos.db.connection import get_connection, check_connection
print(check_connection().split(",")[0])

PostgreSQL 16.15 on x86_64-pc-linux-musl


## 1. What is registered

In [2]:
from services.chunking.engine import ChunkingEngine, available_strategies
from services.chunking.tokenizer import available_tokenizers, get_tokenizer
from services.chunking.models import ChunkingConfig

print("strategies:", available_strategies())
print("tokenizers :", available_tokenizers())

cfg = ChunkingConfig(max_tokens=512, min_tokens=64, overlap_tokens=0)
print("config     :", cfg.to_dict())
print("fingerprint:", cfg.fingerprint())

engine = ChunkingEngine()
print("engine     :", engine.name, "v" + engine.version)

strategies: ['structure_recursive']
tokenizers : ['character', 'simple']
config     : {'max_tokens': 512, 'min_tokens': 64, 'overlap_tokens': 0, 'tokenizer': 'simple', 'include_path_prefix': False}
fingerprint: 2c07e8f12a28778f
engine     : structure_recursive v1.0.0


## 2. Structure decides the boundaries

In [3]:
from services.processing.canonical import (
    CanonicalDocument, Node, NodeType, ProcessorInfo, SourceInfo, Table)

_n = [0]
def node(t, text="", children=None, table=None, attrs=None):
    _n[0] += 1
    return Node(id=f"n-{_n[0]:03d}", type=t, text=text,
                children=children or [], table=table, attributes=attrs or {})

def sec(title, *kids, level=1):
    return node(NodeType.SECTION, title, list(kids), attrs={"level": level})
def para(t): return node(NodeType.PARAGRAPH, t)
def doc(*c):
    return CanonicalDocument(
        source=SourceInfo(path="demo", file_name="demo.htm", doc_format="HTML"),
        processor=ProcessorInfo(name="html", version="2.0.0"), content=list(c))

words = lambda n: " ".join(f"w{i}" for i in range(n))

d = doc(sec("Part I",
            sec("Item 1", para(words(60)), level=2),
            sec("Item 2", para(words(60)), level=2)))

for c in engine.run(d, ChunkingConfig(max_tokens=70, min_tokens=0)).chunks:
    print(f"[{c.chunk_index}] {c.token_count:>4} tok  path={c.path}")

[0]   62 tok  path=['Part I', 'Item 1']
[1]   62 tok  path=['Part I', 'Item 2']


In [4]:
# Same document, a budget big enough to hold Part I whole -> one chunk.
for c in engine.run(d, ChunkingConfig(max_tokens=500, min_tokens=0)).chunks:
    print(f"[{c.chunk_index}] {c.token_count:>4} tok  path={c.path}  nodes={c.node_types()}")

[0]  126 tok  path=['Part I']  nodes=['section']


## 3. Tables split by rows, never by tokens

In [5]:
rows = [[f"FY20{i:02d}", f"{i*1000:,}", f"{i*37:,}"] for i in range(1, 31)]
big_table = node(NodeType.TABLE,
                 table=Table(rows=rows, header=["Year", "Revenue", "Income"],
                             caption="Selected financial data"))

chunks = engine.run(doc(big_table), ChunkingConfig(max_tokens=120, min_tokens=0)).chunks
print(f"{len(chunks)} parts\n")
for c in chunks:
    t = c.tables()[0].table
    print(f"part {c.metadata['table_part']}/{c.metadata['table_parts']} "
          f"{c.token_count:>4} tok  rows={t.n_rows}  header={t.header}  caption={t.caption!r}")

recovered = [r for c in chunks for r in c.tables()[0].table.rows]
print("\nevery row preserved exactly once:", recovered == rows)

2 parts

part 1/2  118 tok  rows=22  header=['Year', 'Revenue', 'Income']  caption='Selected financial data'
part 2/2   48 tok  rows=8  header=['Year', 'Revenue', 'Income']  caption='Selected financial data'

every row preserved exactly once: True


## 4. Token budget and overlap

In [6]:
d2 = doc(sec("A", *[para(words(30)) for _ in range(12)]))

for kw in [dict(max_tokens=100, overlap_tokens=0),
           dict(max_tokens=100, overlap_tokens=25),
           dict(max_tokens=300, overlap_tokens=0)]:
    cfg2 = ChunkingConfig(min_tokens=0, **kw)
    cs = engine.run(d2, cfg2).chunks
    tot = sum(c.token_count for c in cs)
    print(f"max={kw['max_tokens']:<4} overlap={kw['overlap_tokens']:<3} "
          f"-> {len(cs):>2} chunks, {tot:>5} tokens total, "
          f"max={max(c.token_count for c in cs)}  cfg={cfg2.fingerprint()}")

max=100  overlap=0   ->  4 chunks,   360 tokens total, max=90  cfg=707aab74e70e6e2b
max=100  overlap=25  ->  4 chunks,   435 tokens total, max=115  cfg=b656618d6b56b54c
max=300  overlap=0   ->  2 chunks,   360 tokens total, max=300  cfg=17fdf31d487e70ed


## 5. A real SEC filing

In [7]:
from services.processing.storage import read_canonical

with get_connection(autocommit=True) as conn:
    row = conn.execute("""
        SELECT id, file_name, processed_path FROM documents
         WHERE status='PROCESSED' AND file_name LIKE '%tsla%'
         ORDER BY file_name DESC LIMIT 1""").fetchone()

canon = read_canonical(row["processed_path"])
result = engine.run(canon, ChunkingConfig())

print(row["file_name"])
print("canonical nodes:", canon.stats()["nodes"])
print("chunk stats    :", result.stats())
print()
for c in result.chunks[8:16]:
    print(f"[{c.chunk_index:>3}] {c.token_count:>4} tok "
          f"{'T' if c.has_table() else ' '}  {' > '.join(c.path)[:60]}")

2026-01-29_0001628280-26-003952_tsla-20251231.htm
canonical nodes: 1116
chunk stats    : {'chunks': 278, 'tokens_total': 74452, 'tokens_min': 1, 'tokens_max': 512, 'tokens_avg': 267.8, 'with_tables': 65, 'over_budget': 0}

[  8]  401 tok    Tesla, Inc. > Forward-Looking Statements
[  9]  258 tok    Tesla, Inc. > Overview
[ 10]  126 tok    Tesla, Inc. > Segment Information
[ 11]  370 tok    Tesla, Inc. > Automotive
[ 12]  177 tok    Tesla, Inc. > Energy Storage Products
[ 13]   71 tok    Tesla, Inc. > Energy Generation Offerings
[ 14]  208 tok    Tesla, Inc. > Self-Driving Development and Artificial Intell
[ 15]   85 tok    Tesla, Inc. > Vehicle Control and Infotainment Software


## 6. What is stored in PostgreSQL

In [8]:
with get_connection(autocommit=True) as conn:
    print("row counts")
    for t in ("documents", "document_sections", "document_chunks"):
        n = conn.execute(f"SELECT count(*) n FROM {t}").fetchone()["n"]
        print(f"  {t:<20}{n:>9,}")

    print("\nchunk sets")
    for r in conn.execute("""
        SELECT strategy, strategy_version, config_hash, count(*) chunks,
               count(DISTINCT document_id) docs, round(avg(token_count),1) avg
          FROM document_chunks GROUP BY 1,2,3 ORDER BY 1,2,3""").fetchall():
        print(f"  {r['strategy']} v{r['strategy_version']} cfg={r['config_hash']} "
              f"docs={r['docs']:<4} chunks={r['chunks']:<7} avg={r['avg']}")

row counts
  documents                  45
  document_sections      10,287
  document_chunks        13,523

chunk sets
  structure_recursive v1.0.0 cfg=2c07e8f12a28778f docs=45   chunks=11873   avg=262.3
  structure_recursive v1.0.0 cfg=37c3a54c132b21a0 docs=3    chunks=889     avg=289.9
  structure_recursive v1.0.0 cfg=f0615e9a52c2d591 docs=3    chunks=761     avg=265.1


## 7. One chunk, with lineage

In [9]:
with get_connection(autocommit=True) as conn:
    c = conn.execute("""
        SELECT chunk_index, token_count, char_count, has_table, node_path,
               node_ids, section_node_id, content, metadata, content_nodes
          FROM document_chunks
         WHERE document_id=%s AND has_table AND token_count BETWEEN 200 AND 500
         ORDER BY chunk_index LIMIT 1""", (row["id"],)).fetchone()

print("chunk        ", c["chunk_index"])
print("tokens/chars ", c["token_count"], "/", c["char_count"])
print("path         ", " > ".join(c["node_path"]))
print("section node ", c["section_node_id"])
print("canonical    ", ", ".join(c["node_ids"][:6]))
print("metadata     ", {k: v for k, v in c["metadata"].items() if k != "node_types"})

tbls = [n for n in c["content_nodes"] if n["type"] == "table"]
print(f"\nstructured tables kept: {len(tbls)}")
for t in tbls[:1]:
    print("  header:", t["table"]["header"])
    for r_ in t["table"]["rows"][:3]:
        print("   ", r_)

print("\ntext preview:\n", c["content"][:400])

chunk         4
tokens/chars  497 / 2498
path          Tesla, Inc. > Securities registered pursuant to Section 12(b) of the Act:
section node  n-0018
canonical     n-0019, n-0020, n-0021, n-0022, n-0023, n-0024
metadata      {'path_depth': 2}

structured tables kept: 3
  header: []
    ['Title of each class', 'Trading Symbol(s)', 'Name of each exchange on which registered']
    ['Common stock', 'TSLA', 'The Nasdaq Global Select Market']

text preview:
 Title of each class | Trading Symbol(s) | Name of each exchange on which registered

Common stock | TSLA | The Nasdaq Global Select Market

Securities registered pursuant to Section 12(g) of the Act:

None

Indicate by check mark whether the registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act. Yes x No o

Indicate by check mark if the registrant is not required


## 8. Lineage back to company

In [10]:
with get_connection(autocommit=True) as conn:
    r = conn.execute("""
        SELECT ch.chunk_index, array_to_string(ch.node_path,' > ') AS path,
               d.file_name, f.form_type, f.accession_number, co.name, co.cik, co.ticker
          FROM document_chunks ch
          JOIN documents d  ON d.id = ch.document_id
          JOIN filings f    ON f.id = d.filing_id
          JOIN companies co ON co.id = f.company_id
         WHERE ch.document_id=%s ORDER BY ch.chunk_index LIMIT 1""",
        (row["id"],)).fetchone()
for k, v in r.items():
    print(f"  {k:<18}{v}")

  chunk_index       0
  path              SECURITIES AND EXCHANGE COMMISSION
  file_name         2026-01-29_0001628280-26-003952_tsla-20251231.htm
  form_type         10-K
  accession_number  0001628280-26-003952
  name              Tesla, Inc.
  cik               0001318605
  ticker            TSLA


## 9. Idempotency

In [11]:
from services.chunking.pipeline import chunk_pending
import uuid as _uuid

s = chunk_pending(document_id=row["id"])
print("->", s.summary())

INFO    chunking.pipeline    Chunking stage starting | strategy=structure_recursive v1.0.0 tokenizer=simple max_tokens=512 overlap=0 cfg=2c07e8f12a28778f


INFO    chunking.pipeline    1 document(s) to consider


INFO    chunking.pipeline    Skipping (already chunked by structure_recursive v1.0.0 cfg=2c07e8f12a28778f, 278 chunks): 2026-01-29_0001628280-26-003952_tsla-20251231.htm


INFO    chunking.pipeline    Chunking stage complete | chunked=0 skipped=1 failed=0 chunks=0


-> chunked=0 skipped=1 failed=0 chunks=0


## 10. A different config is a new chunk set, not an overwrite

In [12]:
s = chunk_pending(document_id=row["id"], config=ChunkingConfig(max_tokens=256))
print("->", s.summary())

with get_connection(autocommit=True) as conn:
    for r in conn.execute("""
        SELECT config_hash, config->>'max_tokens' AS max_tokens, count(*) chunks
          FROM document_chunks WHERE document_id=%s
         GROUP BY 1,2 ORDER BY 2""", (row["id"],)).fetchall():
        print(f"  cfg={r['config_hash']} max_tokens={r['max_tokens']:<5} chunks={r['chunks']}")

INFO    chunking.pipeline    Chunking stage starting | strategy=structure_recursive v1.0.0 tokenizer=simple max_tokens=256 overlap=0 cfg=87ff122858a3ab30


INFO    chunking.pipeline    1 document(s) to consider


INFO    chunking.pipeline    Job created: CHUNKING status=QUEUED for 2026-01-29_0001628280-26-003952_tsla-20251231.htm


INFO    chunking.pipeline    Job PROCESSING: CHUNKING 2026-01-29_0001628280-26-003952_tsla-20251231.htm


INFO    chunking.pipeline    Chunked 2026-01-29_0001628280-26-003952_tsla-20251231.htm | chunks=463 tokens avg=159.7 min=1 max=256 with_tables=92 over_budget=0


INFO    chunking.pipeline    Job COMPLETED: CHUNKING 2026-01-29_0001628280-26-003952_tsla-20251231.htm


INFO    chunking.pipeline    Chunking stage complete | chunked=1 skipped=0 failed=0 chunks=463


-> chunked=1 skipped=0 failed=0 chunks=463


  cfg=87ff122858a3ab30 max_tokens=256   chunks=463
  cfg=2c07e8f12a28778f max_tokens=512   chunks=278


## 11. Not implemented yet

In [13]:
with get_connection(autocommit=True) as conn:
    for t in ("document_chunks", "vector_index_records"):
        n = conn.execute(f"SELECT count(*) n FROM {t}").fetchone()["n"]
        print(f"  {t:<22}{n:>9,}")
print("\nno embeddings, no Qdrant collections, no retrieval - next stage")

  document_chunks          13,986
  vector_index_records          0

no embeddings, no Qdrant collections, no retrieval - next stage
